# Curriculum 02 · Lab 2 — Cosine Similarity vs Raw Dot Product

**Goal:** Prove the identity cos(a, b) = a·b for L2-normalized BGE embeddings,
then show what happens when the normalization is missing. Every vector store
ranks passages by a similarity score, and the two most common scores are the
cosine similarity and the raw dot product. Cosine measures the ANGLE between
two vectors and ignores their magnitude; the raw dot product keeps magnitude,
so a long vector scores higher than a short one even when it points in a worse
direction. BGE is trained to produce L2-normalized vectors (every vector has
length 1), so the two scores collapse into one — this lab proves that identity
numerically on real embeddings, then deliberately scales the passage vectors by
a deterministic, length-correlated factor (simulating an embedder that does not
normalize, like E5) and watches the raw dot product mis-rank the results cosine
gets right.

```
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU, L2-normalized)
Data        : rag-mini-wikipedia (passages.parquet + test.parquet, fresh)
Metric      : cosine vs normalized dot (identity) vs raw dot (mis-ranking)
Comparison : Spearman rank corr, top-5 overlap, concrete mis-rank example
```

**Protocol:** take the first 50 passages and 5 questions from the fresh
rag-mini-wikipedia parquet files, embed everything with BGE, build the three
score matrices (cosine, normalized dot, deliberately raw dot on length-scaled
vectors), and compare the rankings they produce.


## 0 · Setup — environment, imports & repo paths

**WHAT:** Silences the model-download progress bar, imports every dependency
(numpy, pandas, the repo's `BGEEmbedding`, optional scipy), and puts the
repo-root component library on `sys.path` so this notebook reuses
`src/embeddings/bge.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE via `sentence-transformers` —
no API embeddings anywhere. The env var `HF_HUB_DISABLE_PROGRESS_BARS` must be
set **before any third-party import** (numpy/pandas already pull in
`huggingface_hub`, which reads the flag at import time). scipy is optional:
`spearman_between` falls back to a pandas implementation if it is missing.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script,
`python src/curriculum/02-embeddings/02-cosine-vs-dot.py`) or from the notebook's
own folder (the Jupyter default) — and `cd`s into it so every path stays
repo-relative.

**WHAT TO EXPECT:** no output — just a clean import. The model itself is loaded
lazily later.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   pandas                -> reads the passages.parquet corpus
#   numpy                 -> score matrices (cosine / dot)
#   scipy                 -> Spearman rank correlation (optional; pandas fallback)
%pip install sentence-transformers pandas numpy scipy



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

# Silence the "Loading weights" progress bar (transformers honors this flag;
# must be set before any third-party import pulls in huggingface_hub).
import os

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402

try:
    from scipy.stats import spearmanr  # noqa: F401

    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False


## 1 · Configuration — the experiment's knobs

**WHAT:** The module-level constants that define the experiment: which corpus
and test files to read, how many passages/questions to use, the top-k, and the
preview width for one-line passage printing.

**WHY:** These are the knobs you tweak to re-run the experiment. `N_PASSAGES =
50` and `N_QUESTIONS = 5` are deterministic heads of the 3200-passage corpus and
`test.parquet` — small enough to keep CPU embedding time reasonable while still
giving the identity proof and the mis-ranking demo real data to chew on.


In [3]:
# --- 1. Configuration — tweak these to rerun the comparison ---------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 50  # deterministic head of the 3200-passage corpus (keeps runtime low)
N_QUESTIONS = 5  # deterministic head of test.parquet
TOP_K = 5
PREVIEW = 62  # max characters of passage text shown next to each hit
MODEL_NAME = "BAAI/bge-base-en-v1.5"


## 2 · Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

**WHAT:** Two small functions: `load_passages` reads the first `n` passages
from `passages.parquet` and returns `(passage_texts, passage_ids)`;
`load_questions` reads the first `n` question strings from `test.parquet`.

**WHY:** The parquet files are the fresh rag-mini-wikipedia corpus — the same
data every track-02 embeddings lab uses, so results stay comparable. The head
of the corpus is deterministic, so the subset is reproducible.


In [4]:
# --- 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, n: int) -> list[str]:
    """Return the first ``n`` question strings from test.parquet."""
    df = pd.read_parquet(path)
    return df["question"].head(n).tolist()


## 3 · Score matrices — cosine, normalized dot, and deliberately raw dot

**WHAT:** The three score matrices at the heart of the lab.
`cosine_similarity_matrix` writes out the formula explicitly — divide each
row/column by its L2 norm, then dot; `normalized_dot_matrix` is the raw dot
product, valid because BGE already L2-normalizes every vector;
`unnormalized_passages` simulates an embedder that skips normalization by
scaling each passage vector by a deterministic length-correlated factor;
`raw_dot_matrix` is the dot product on those deliberately unnormalized vectors.

**WHY:** If the embedder's contract holds (||a|| = ||b|| = 1), the cosine and
normalized-dot matrices must be numerically equal — the identity this lab
proves. The raw-dot matrix on scaled vectors shows what breaks when magnitude
carries signal.


In [5]:
# --- 3. Score matrices — cosine, normalized dot, and deliberately raw dot --
def cosine_similarity_matrix(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """cos(a, b) = (a . b) / (||a|| * ||b||) for every (query, passage) pair.

    Written out explicitly (rather than via sklearn) so the formula is
    visible: each row/column is divided by its L2 norm before the dot product.
    """
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    Pn = P / np.linalg.norm(P, axis=1, keepdims=True)
    return Qn @ Pn.T


def normalized_dot_matrix(Q: np.ndarray, P: np.ndarray) -> np.ndarray:
    """Raw dot product, valid because BGE already L2-normalizes every vector.

    If the embedder's contract holds (||a|| = ||b|| = 1), this must equal the
    cosine matrix above — the identity this lab proves numerically.
    """
    return Q @ P.T


def unnormalized_passages(P: np.ndarray, lengths: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """Simulate an embedder that does NOT normalize its output.

    BGE normalizes every vector to unit length, so magnitude carries no
    signal. Real embedders that skip normalization (e.g. E5) produce vectors
    whose magnitude grows with passage length. We reproduce that effect by
    scaling each passage vector by a deterministic factor derived from its
    character length — the raw dot product then ranks by magnitude * direction
    instead of direction alone. Returns (scaled_vectors, scale_factors).
    """
    scale = np.asarray([1.0 + (length % 200) / 100.0 for length in lengths])
    return P * scale[:, np.newaxis], scale


def raw_dot_matrix(Q: np.ndarray, P_raw: np.ndarray) -> np.ndarray:
    """Raw dot product on the deliberately unnormalized passage vectors."""
    return Q @ P_raw.T


## 4 · Ranking helpers

**WHAT:** Four small helpers: `top_k_indices` returns the top-k passage indices
for one score row; `spearman_between` measures the rank correlation between two
score rows (scipy when available, pandas fallback otherwise); `topk_overlap` is
the order-insensitive fraction of top-k items shared by two orderings;
`preview` flattens a passage for one-line printing.

**WHY:** These turn the three score matrices into the comparison the lab
reports: how much the raw-dot ranking disagrees with the cosine ranking, and
where exactly it mis-ranks.


In [6]:
# --- 4. Ranking helpers ---------------------------------------------------
def top_k_indices(scores: np.ndarray, k: int) -> list[int]:
    """Indices of the top-k passages for one query's score row."""
    return np.argsort(scores)[::-1][:k].tolist()


def spearman_between(cos_scores: np.ndarray, raw_scores: np.ndarray) -> float:
    """Spearman rank correlation between two score rows (per query)."""
    cos_ranks = np.argsort(np.argsort(cos_scores))
    raw_ranks = np.argsort(np.argsort(raw_scores))
    if HAVE_SCIPY:
        return float(spearmanr(cos_ranks, raw_ranks).statistic)
    return float(pd.Series(cos_ranks).corr(pd.Series(raw_ranks), method="spearman"))


def topk_overlap(a: list[int], b: list[int]) -> float:
    """Fraction of top-k items shared by two orderings (order-insensitive)."""
    return len(set(a) & set(b)) / len(a)


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 5 · Run — [1] setup printout, load, embed

**WHAT:** The first steps of the experiment: load the 50 passages and 5
questions, print the setup block, and embed everything with BGE.

**WHY:** The setup printout makes the run self-describing; the embedding step
is where the model is actually loaded (cached locally — no download on this
run).


In [7]:
# --- 5. Print the artifact — runnable demo --------------------------------
# --- 2. Load --------------------------------------------------------------
passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
questions = load_questions(TEST_PATH, N_QUESTIONS)

print("=" * 66)
print("Lab 02 — cosine similarity vs raw dot product for retrieval")
print(f"model: {MODEL_NAME} (local BGE, L2-normalized embeddings)")
print("=" * 66)

print(f"\n[1] Setup")
print(f"    {len(passage_texts)} passages (first {N_PASSAGES} of 3200, "
      f"ids {passage_ids[0]}..{passage_ids[-1]})")
print(f"    {len(questions)} questions from test.parquet (first {N_QUESTIONS}):")
for q in questions:
    print(f"      - {q}")
print(f"    Spearman via {'scipy' if HAVE_SCIPY else 'pandas fallback'}")

# --- 3. Embed -------------------------------------------------------------
embedder = BGEEmbedding(model_name=MODEL_NAME)
P = np.asarray(embedder.embed_documents(passage_texts), dtype=np.float32)
Q = np.asarray([embedder.embed_query(q) for q in questions], dtype=np.float32)
print(f"\n    embedded {P.shape[0]} passages x {P.shape[1]} dims, "
      f"{Q.shape[0]} queries x {Q.shape[1]} dims")


Lab 02 — cosine similarity vs raw dot product for retrieval
model: BAAI/bge-base-en-v1.5 (local BGE, L2-normalized embeddings)

[1] Setup
    50 passages (first 50 of 3200, ids 0..49)
    5 questions from test.parquet (first 5):
      - Was Abraham Lincoln the sixteenth President of the United States?
      - Did Lincoln sign the National Banking Act of 1863?
      - Did his mother die of pneumonia?
      - How many long was Lincoln's formal education?
      - When did Lincoln begin his political career?
    Spearman via scipy



    embedded 50 passages x 768 dims, 5 queries x 768 dims


## 6 · [2] Cosine vs normalized dot — the identity proof

**WHAT:** Build the three score matrices, then compare the cosine matrix with
the normalized-dot matrix and print the maximum absolute difference over all
scores.

**WHY:** This is the numerical proof of the identity cos(a, b) = a·b for
L2-normalized vectors. The max difference should be ~1e-7 (float32 precision) —
for a normalized embedder, cosine similarity IS the dot product.


In [8]:
# --- 4. Three score matrices ----------------------------------------------
cosine = cosine_similarity_matrix(Q, P)
norm_dot = normalized_dot_matrix(Q, P)
P_raw, scale = unnormalized_passages(P, [len(t) for t in passage_texts])
raw_dot = raw_dot_matrix(Q, P_raw)

# --- [2] cosine vs normalized dot: the identity ---------------------------
max_diff = float(np.max(np.abs(cosine - norm_dot)))
print("\n[2] Cosine vs normalized dot — the identity proof")
print("    cos(a,b) = (a.b)/(||a||*||b||); BGE guarantees ||a||=||b||=1,")
print("    so cos(a,b) = a.b. The two matrices must be numerically equal:")
print(f"    max |cosine - normalized_dot| over all {cosine.size} scores = {max_diff:.3e}")
print("    -> identical to within float32 precision. For a normalized")
print("       embedder, cosine similarity IS the dot product.")



[2] Cosine vs normalized dot — the identity proof
    cos(a,b) = (a.b)/(||a||*||b||); BGE guarantees ||a||=||b||=1,
    so cos(a,b) = a.b. The two matrices must be numerically equal:
    max |cosine - normalized_dot| over all 250 scores = 1.192e-07
    -> identical to within float32 precision. For a normalized
       embedder, cosine similarity IS the dot product.


## 7 · [3] Cosine vs raw dot — the disagreement

**WHAT:** Compare the cosine ranking with the raw-dot ranking on the
deliberately unnormalized passage vectors: mean Spearman rank correlation, mean
top-5 overlap, how many queries keep an identical top-5 order — then print one
concrete mis-ranking example (first query whose top-1 differs) as a side-by-side
rank table.

**WHY:** This is the cautionary half of the lab. Raw dot ranks by magnitude *
direction instead of direction alone, so a longer passage can outrank a
better-direction one. The concrete example makes the failure visible.


In [9]:
# --- [3] cosine vs raw dot: the disagreement ------------------------------
print("\n[3] Cosine vs raw dot on deliberately unnormalized vectors")
print("    Passage vectors were scaled by a deterministic length-correlated")
print("    factor (1.0..2.99) to simulate an embedder that skips L2")
print("    normalization. Raw dot then ranks by magnitude * direction.")
rho_list, overlap_list, identical_list = [], [], []
for i in range(len(questions)):
    cos_top = top_k_indices(cosine[i], TOP_K)
    raw_top = top_k_indices(raw_dot[i], TOP_K)
    rho_list.append(spearman_between(cosine[i], raw_dot[i]))
    overlap_list.append(topk_overlap(cos_top, raw_top))
    identical_list.append(cos_top == raw_top)
print(f"    Spearman(cosine ranks, raw-dot ranks), mean over "
      f"{len(questions)} queries: {np.mean(rho_list):.3f}")
print(f"    top-{TOP_K} overlap (shared items, order-insensitive), mean: "
      f"{np.mean(overlap_list):.2f}")
print(f"    queries where the top-{TOP_K} order is identical: "
      f"{sum(identical_list)}/{len(questions)}")

# Concrete mis-ranking example: first query whose top-1 differs.
demo = next(
    (
        i
        for i in range(len(questions))
        if top_k_indices(cosine[i], 1) != top_k_indices(raw_dot[i], 1)
    ),
    0,
)
cos_top = top_k_indices(cosine[demo], TOP_K)
norm_top = top_k_indices(norm_dot[demo], TOP_K)
raw_top = top_k_indices(raw_dot[demo], TOP_K)
print(f"\n    Concrete example — question {demo}:")
print(f'      "{questions[demo]}"')
print(f"      {'rank':<5}{'cosine':>8}{'norm_dot':>10}{'raw_dot':>8}"
      f"   (passage ids into the {N_PASSAGES}-passage subset)")
for r in range(TOP_K):
    print(f"      {r + 1:<5}{cos_top[r]:>8}{norm_top[r]:>10}{raw_top[r]:>8}")
print("    cosine and normalized_dot return the identical top-5; raw_dot")
print("    reorders it. The raw-dot winner is a longer passage whose")
print(f"    magnitude factor {scale[raw_top[0]]:.2f} outweighs its worse direction:")
print(f"      raw_dot #1 (id {raw_top[0]}): {preview(passage_texts[raw_top[0]])}")
print(f"      cosine  #1 (id {cos_top[0]}): {preview(passage_texts[cos_top[0]])}")



[3] Cosine vs raw dot on deliberately unnormalized vectors
    Passage vectors were scaled by a deterministic length-correlated
    factor (1.0..2.99) to simulate an embedder that skips L2
    normalization. Raw dot then ranks by magnitude * direction.
    Spearman(cosine ranks, raw-dot ranks), mean over 5 queries: 0.277
    top-5 overlap (shared items, order-insensitive), mean: 0.20
    queries where the top-5 order is identical: 0/5

    Concrete example — question 0:
      "Was Abraham Lincoln the sixteenth President of the United States?"
      rank   cosine  norm_dot raw_dot   (passage ids into the 50-passage subset)
      1          17        17      22
      2           5         5      13
      3           2         2      47
      4          12        12      21
      5          13        13      39
    cosine and normalized_dot return the identical top-5; raw_dot
    reorders it. The raw-dot winner is a longer passage whose
    magnitude factor 2.92 outweighs its worse direc

## 8 · Takeaway — [4] what the comparison teaches

**WHAT:** Print the lesson the comparison teaches.

**WHY:** Cosine similarity is the standard for text embeddings because it
isolates direction (meaning) from magnitude (length), which is usually noise.
The raw dot product is only equivalent when the embedder guarantees unit-norm
vectors — BGE does, so a dot-product vector store is safe with BGE. If you swap
in an embedder that does not normalize (e.g. E5), either normalize the vectors
yourself or switch the store's metric to cosine.


In [10]:
# --- [4] Takeaway ---------------------------------------------------------
print("\n[4] Takeaway")
print("    Cosine similarity is the standard for text embeddings because")
print("    it isolates direction (meaning) from magnitude (length), which")
print("    is usually noise. The raw dot product is only equivalent when")
print("    the embedder guarantees unit-norm vectors — BGE does, so a")
print("    dot-product vector store is safe with BGE. If you swap in an")
print("    embedder that does not normalize (e.g. E5), either normalize")
print("    the vectors yourself or switch the store's metric to cosine.")



[4] Takeaway
    Cosine similarity is the standard for text embeddings because
    it isolates direction (meaning) from magnitude (length), which
    is usually noise. The raw dot product is only equivalent when
    the embedder guarantees unit-norm vectors — BGE does, so a
    dot-product vector store is safe with BGE. If you swap in an
    embedder that does not normalize (e.g. E5), either normalize
    the vectors yourself or switch the store's metric to cosine.
